# Motivation：VLM 输入 × 机器人速度 × DVFS

**问题背景**：机器人上只跑一个 VLM（Qwen2.5-VL-3B，Q4_K_M，Jetson Orin NX 16GB），没有其他应用。

- **任务给定的输入**：图像分辨率、图片数、文本长度，以及机器人能动作之前需要的输出 token 数 `n_act`。
- **任务 / 规划器给定的量**：速度 v。
- **我们能控制的**：只有 DVFS，即 CPU × GPU 频率。

这个 notebook 只用已有的测量数据回答一个问题：**我们的 objective 应该是什么，数据能不能支撑它。**

**数据**：
- `data/vlm_matrix.csv`：4 种分辨率 × 3 种图片数 × 3 种文本长度，在 3 个 CPU 频率 × 7 个 GPU 频率上测，外加默认 governor。每个配置重复 5 次，共 3960 个请求。
- `data/vlm_matrix_idle.csv`：每个频率点的空闲功率。
- 由 `vlm_bench.py` 测得。

**符号**：
- L_q(n)：前 n 个输出 token 完成时间的 p95，即 TTA(n) 的 p95。
- v*：满足安全约束的最大速度（定义见 Obs 4）。

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import plot_style; plot_style.apply()            # Times New Roman 16 pt; 2 panels (8,4), 3 panels (15,4)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = Path("data")
df = pd.read_csv(DATA / "vlm_matrix.csv")
idle = pd.read_csv(DATA / "vlm_matrix_idle.csv")
df["res"] = df.res_w.astype(str) + "x" + df.res_h.astype(str)
df["inp"] = df.res + " x" + df.n_img.astype(str) + " t" + df.n_txt_target.astype(str)

# safety / task parameters (assumptions, same defaults as vlm_speed_requirements.py)
D, A_BRAKE, T_ACT, MARGIN, Q = 5.0, 2.0, 0.05, 0.5, 0.95   # m, m/s^2, s, m, latency quantile
DQ = 2.0                                                    # event-triggered: one query per DQ metres
NACT = [8, 16, 32]
V = np.round(np.arange(0.25, 3.01, 0.25), 2)

INP = ["res_w", "res", "n_img", "n_txt_target", "inp"]
g = df.groupby(["cpu", "gpu"] + INP)
agg = g.median(numeric_only=True)
for n in NACT:
    agg[f"q_{n}"] = g[f"tta_{n}"].quantile(Q)
agg = agg.reset_index()


def v_star(tta_ms):
    T = np.asarray(tta_ms, float) / 1000 + T_ACT
    r = D - MARGIN
    return A_BRAKE * (-T + np.sqrt(T ** 2 + 2 * r / A_BRAKE))


for n in NACT:
    agg[f"v_{n}"] = v_star(agg[f"q_{n}"])
P_IDLE = idle.groupby(["cpu", "gpu"]).idle_w.median()
agg["p_idle"] = [P_IDLE.get((c, gq), np.nan) for c, gq in zip(agg.cpu, agg.gpu)]
gov = agg[(agg.cpu == 0) & (agg.gpu == 0)].set_index("inp")
fx = agg[agg.cpu > 0]
CPUS, GPUS = sorted(fx.cpu.unique()), sorted(fx.gpu.unique())
CMAX, GMAX = CPUS[-1], GPUS[-1]
top = fx[(fx.cpu == CMAX) & (fx.gpu == GMAX)].set_index("inp")
ORDER = top.sort_values("n_vis").index.tolist()
print(f"{len(df)} requests, {df.groupby(INP).ngroups} inputs, CPU {CPUS}, GPU {GPUS} MHz (+ stock governor)")

## Obs 1：输入不同，TTFT 相差一个数量级以上，主要由视觉 token 数决定

In [ ]:
t = top.loc[ORDER]
print(f"TTFT @ {CMAX}/{GMAX} MHz: {t.ttft.min():.0f} - {t.ttft.max():.0f} ms  (x{t.ttft.max() / t.ttft.min():.1f})")
print(f"TTA_p95(32) @ max clocks: {t.q_32.min():.0f} - {t.q_32.max():.0f} ms")
print(f"TPOT: {t.tpot.min():.0f} - {t.tpot.max():.0f} ms  -> output length adds ~n_act x 52 ms regardless of input")

fig, ax = plt.subplots(1, 2, figsize=plot_style.FIG2, gridspec_kw=dict(width_ratios=[1.5, 1]))
s = t[t.n_txt_target == 128]
bottom = np.zeros(len(s))
for c, lab in [("t_pre", "preprocess"), ("t_enc", "vision encoder"), ("t_img_pf", "image prefill"),
               ("t_txt_pf", "text prefill")]:
    ax[0].bar(range(len(s)), s[c] / 1000, bottom=bottom, label=lab)
    bottom += s[c].values / 1000
ax[0].set_xticks(range(len(s)), [f"{r.res_w}×{r.n_img}" for r in s.itertuples()], rotation=60, ha="right")
ax[0].set(ylabel="TTFT [s]", xlabel="width × #images", title="(a) TTFT breakdown, text 128")
ax[0].legend(frameon=False, labelspacing=.2, handlelength=1)
for k, d in t.groupby("n_txt_target"):
    ax[1].scatter(d.n_vis, d.ttft / 1000, s=14, label=f"text {k}")
ax[1].set(xlabel="visual tokens $N_{vis}$", ylabel="TTFT [s]", title="(b) TTFT vs. $N_{vis}$")
ax[1].legend(frameon=False)
for a_ in ax: a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

**结论**：
- 视觉部分（编码器加图像预填充）占了 TTFT 的大头。
- 分辨率 × 图片数，也就是视觉 token 数，基本决定了 TTFT。例如 224×168 × 4 图和 448×336 × 1 图都是 192 个视觉 token，TTFT 几乎一样。
- 文本长度是第二个因素：从 32 加到 512 token，多 0.6–1 s。
- **输入由任务决定，不能拿来优化，但它决定了延迟从哪里起步。**

## Obs 2：延迟 ≈ a + b / f_GPU，CPU 频率几乎没有影响

In [ ]:
fits = []
for k, d in fx[fx.cpu == CMAX].groupby("inp"):
    x = 1000 / d.gpu.values
    b, a0 = np.polyfit(x, d.ttft.values, 1)
    fits.append(dict(inp=k, a=a0, b=b, R2=np.corrcoef(x, d.ttft)[0, 1] ** 2,
                     speedup=d.ttft.max() / d.ttft.min()))
fits = pd.DataFrame(fits).set_index("inp").loc[ORDER]
print(f"TTFT = a + b*1000/f fit: R2 min {fits.R2.min():.3f}; GPU 306 -> 918 MHz speeds up TTFT "
      f"x{fits.speedup.min():.1f} - x{fits.speedup.max():.1f}")
c = fx[fx.gpu == GMAX].pivot_table(index="inp", columns="cpu", values="ttft")
r = c[CPUS[0]] / c[CMAX]
print(f"CPU {CPUS[0]} vs {CMAX} MHz at GPU {GMAX}: TTFT ratio {r.min():.2f} - {r.max():.2f}")

fig, ax = plt.subplots(1, 2, figsize=plot_style.FIG2)
for k in [ORDER[0], ORDER[len(ORDER) // 2], ORDER[-1]]:
    for cpu, ls in zip(CPUS, [":", "--", "-"]):
        d = fx[(fx.inp == k) & (fx.cpu == cpu)]
        ax[0].plot(d.gpu, d.ttft / 1000, ls, marker="o", ms=3, label=k if cpu == CMAX else None)
ax[0].set(xlabel="GPU [MHz]", ylabel="TTFT [s]", yscale="log", ylim=(0.3, 400), title="(a) TTFT vs. clocks")
ax[0].legend(frameon=False)
ax[1].bar(range(len(r)), r.loc[ORDER].values)
ax[1].axhline(1, color="k", lw=.8)
ax[1].set(ylim=(0.9, 1.1), xticks=[], xlabel="inputs (sorted by N_vis)",
          ylabel=f"TTFT ratio CPU {CPUS[0]}/{CMAX}", title="(b) CPU effect")
for a_ in ax: a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

**结论**：
- GPU 频率能把延迟调节约 2.4–2.7 倍，是唯一有效的"性能旋钮"。
- CPU 频率对延迟几乎没有影响，但会多耗功率（见 Obs 3）。所以 CPU 应该尽量放低。

## Obs 3：功率随频率变化：工作功率翻倍，空闲功率差 1.4 W，单次请求能耗在高 GPU 频率时最低

In [ ]:
pw = fx.groupby(["cpu", "gpu"]).agg(mean_w=("mean_w", "median"), p_idle=("p_idle", "first")).reset_index()
fx = fx.assign(e_rel=fx.energy_8 / fx.inp.map(top.energy_8))
er = fx.groupby(["cpu", "gpu"]).e_rel.mean().reset_index()

fig, ax = plt.subplots(1, 2, figsize=plot_style.FIG2)
for cpu, ls in zip(CPUS, [":", "--", "-"]):
    d = pw[pw.cpu == cpu]
    ax[0].plot(d.gpu, d.mean_w, ls, marker="o", label=f"CPU {cpu}")
    ax[0].plot(d.gpu, d.p_idle, ls, marker="s", color="gray", ms=3, label="idle" if cpu == CMAX else None)
    e = er[er.cpu == cpu]
    ax[1].plot(e.gpu, e.e_rel, ls, marker="o", label=f"CPU {cpu}")
ax[0].axhline(gov.mean_w.median(), color="r", lw=1, label="governor")
ax[0].set(xlabel="GPU [MHz]", ylabel="board power [W]", ylim=(4, 26), title="(a) active / idle power")
ax[1].set(xlabel="GPU [MHz]", ylabel="rel. energy / request", title="(b) energy per request")
for a_ in ax: a_.grid(alpha=.3)
ax[0].legend(frameon=False, ncol=2, loc="upper left", columnspacing=.8, handlelength=1.5)
ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
print(pw.pivot(index="cpu", columns="gpu", values="mean_w").round(1))
print("energy per request relative to max clocks:\n", er.pivot(index="cpu", columns="gpu", values="e_rel").round(2))

**结论**：功率的两种计法给出相反的最优点。
- **按平均功率**（VLM 一直在跑）：越低频越省，约 8 W 对 17 W。
- **按单次请求能耗**（跑完就空闲）：GPU 在高频时最低，因为跑得快、早结束；CPU 在 576 时最低。

所以 **objective 是"平均功率"还是"单次请求能耗"，取决于 VLM 怎么被调用**，两者的最优频率不同，必须在问题定义里写清楚。

## Obs 4：速度决定延迟上限，不同输入能支持的速度相差约 6 倍

**安全约束**：VLM 给出结果之前机器人继续前进，之后还要能在 D 米内刹停：

v·(L_q(n_act) + t_act) + v²/(2a) + margin ≤ D

**参数**：D = 5 m，a = 2 m/s²，t_act = 0.05 s，margin = 0.5 m。这些都是假设值。

反过来说，**速度 v 定了，延迟上限 τ(v) 也就定了**：τ(v) = (D − margin − v²/2a)/v − t_act。

In [ ]:
def tau(v):   # latency budget [s] implied by speed v
    v = np.asarray(v, float)
    return (D - MARGIN - v ** 2 / (2 * A_BRAKE)) / v - T_ACT


print("latency budget tau(v) [s]:", {float(v): round(float(tau(v)), 2) for v in [0.25, 0.5, 1, 1.5, 2, 3]})
best_cpu = fx.groupby(["inp", "gpu"]).v_8.max().unstack().loc[ORDER]
print(f"v* (n_act=8) at max clocks: {top.v_8.min():.2f} - {top.v_8.max():.2f} m/s; "
      f"at GPU {GPUS[0]}: {best_cpu[GPUS[0]].min():.2f} - {best_cpu[GPUS[0]].max():.2f} m/s")

fig, ax = plt.subplots(1, 2, figsize=plot_style.FIG2, gridspec_kw=dict(width_ratios=[1.2, 1]))
im = ax[0].imshow(best_cpu.values, aspect="auto", cmap="viridis")
ax[0].set_xticks(range(len(GPUS))[::2], GPUS[::2])
YT = [0, 9, 18, 27, 35]
NV = top.loc[ORDER, "n_vis"].astype(int).values
ax[0].set_yticks(YT, [str(NV[i]) for i in YT])
ax[0].set(xlabel="GPU [MHz]", ylabel="inputs (by $N_{vis}$)", title="(a) $v^*$ [m/s], $n_{act}$=8")
plt.colorbar(im, ax=ax[0])
for n in NACT:
    ax[1].plot(top.loc[ORDER, "n_vis"], top.loc[ORDER, f"v_{n}"], "o", ms=3, label=f"$n_{{act}}$={n}")
ax[1].set(xlabel="visual tokens", ylabel="$v^*$ [m/s] at max clocks", title="(b) speed capability")
ax[1].grid(alpha=.3); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**结论**：
- 安全约束把**速度和输入耦合**在一起：输入越大、n_act 越多，同样的速度就需要越高的 GPU 频率。
- 某些（输入，速度）组合，即使用最高频也不安全。这说明**速度本身就是一个硬上限**：v_max(输入) = v*(最高频)。

## Obs 5：默认 governor 不知道速度：高速时延迟不够，低速时又白白耗电

In [ ]:
gv = gov.loc[ORDER]
print(f"stock governor under load: CPU {gv.cpu_mean.median():.0f} MHz, GPU {gv.gpu_mean.median():.0f} MHz (median);"
      f" power {gv.mean_w.min():.1f}-{gv.mean_w.max():.1f} W vs {top.mean_w.min():.1f}-{top.mean_w.max():.1f} W at max clocks")
print(f"governor TTFT / max-clock TTFT: {(gv.ttft / top.loc[ORDER].ttft).min():.2f} - {(gv.ttft / top.loc[ORDER].ttft).max():.2f}")
rows = []
for v in V:
    feas_max = top.v_8 >= v
    feas_gov = gv.v_8 >= v
    if feas_max.sum():
        rows.append(dict(v=v, feasible_at_max=int(feas_max.sum()),
                         governor_misses=int((feas_max & ~feas_gov.reindex(feas_max.index)).sum())))
print(pd.DataFrame(rows).set_index("v").T)

**结论**：
- 默认 governor 在 VLM 负载下基本把 GPU 拉到接近最高频，功率接近最高频时的水平，但延迟又比最高频长。
- 低速时它不需要这么快，白白耗电。高速时，一些最高频本来能满足的（输入，速度）组合，它反而达不到。
- 它**既不知道输入，也不知道速度**。这就是我们要解决的问题。

## Obs 6：最优频率同时取决于输入和速度，任何固定设置都不够

两种调用方式下的平均功率（VLM 在 n_act 个 token 后停止）：
- **连续查询（A）**：请求一个接一个，平均功率 P̄_A(f) = E_req(f) / L(f)。约束只有安全约束 v_n(f) ≥ v。
- **事件触发（B）**：每走 dq 米查询一次，周期 T_q = dq / v。平均功率 P̄_B(f) = [E_req(f) + P_idle(f)·(T_q − L(f))] / T_q。约束是 v_n(f) ≥ v 且 L_q(f) ≤ T_q。

**比较的策略**（都只在最高频能满足约束的（输入，速度）组合上比较）：
- `max`：固定 1984/918 MHz。
- `governor`：默认调频。
- `speed-only`：只知道速度，每个速度选一个频率，要对最坏的输入也满足约束。
- `input-only`：只知道输入，每个输入选一个频率，要对最高可行速度也满足约束。
- `oracle`：同时知道输入和速度。

In [ ]:
N = 8


def cost(rows, v, mode):
    # average power [W] of every DVFS row for speed v; NaN if infeasible
    L, E = rows[f"tta_{N}"].values / 1000, rows[f"energy_{N}"].values
    ok = rows[f"v_{N}"].values >= v
    if mode == "A":
        return np.where(ok, E / L, np.nan)
    Tq = DQ / v
    ok &= rows[f"q_{N}"].values / 1000 <= Tq
    return np.where(ok, (E + rows.p_idle.values * (Tq - L)) / Tq, np.nan)


def policies(mode):
    tab = {}                                  # (inp, v) -> cost per DVFS point
    pts = list(fx.groupby(["cpu", "gpu"]).groups)
    for k, d in fx.groupby("inp"):
        d = d.set_index(["cpu", "gpu"]).loc[pts].reset_index()
        for v in V:
            c = cost(d, v, mode)
            if not np.isnan(c[pts.index((CMAX, GMAX))]):
                tab[(k, v)] = c
    T = pd.DataFrame(tab, index=pd.MultiIndex.from_tuples(pts)).T     # rows (inp, v), cols DVFS
    res = pd.DataFrame(index=T.index)
    res["max"] = T[(CMAX, GMAX)]
    res["governor"] = [float(cost(gov.loc[[k]], v, mode)[0]) for k, v in T.index]
    res["oracle"] = T.min(axis=1)
    res["oracle_gpu"] = [pts[int(np.nanargmin(r))][1] for r in T.values]
    # one point per speed: must be feasible for every input at that speed -> min of the column sums
    for name, lvl in [("speed-only", 1), ("input-only", 0)]:
        best = {}
        for key, sub in T.groupby(level=lvl):
            allok = sub.notna().all()
            best[key] = sub.loc[:, allok].sum().idxmin() if allok.any() else None
        res[name] = [T.loc[(k, v), best[(k, v)[lvl]]] if best[(k, v)[lvl]] else np.nan for k, v in T.index]
    return res


summ = {}
for mode in ["A", "B"]:
    R = policies(mode)
    globals()[f"R_{mode}"] = R
    m = R[["max", "governor", "input-only", "speed-only", "oracle"]].mean()
    summ[f"mode {mode}"] = pd.concat([m.round(2), (100 * (1 - m / m["max"])).round(1).add_prefix("saving% ")])
print(f"mean average power [W] over feasible (input, v) points, n_act={N}, dq={DQ} m")
print(pd.DataFrame(summ))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=plot_style.FIG2, sharey=True, layout="constrained")
for a_, mode in zip(ax, "AB"):
    R = globals()[f"R_{mode}"]
    hm = R.oracle_gpu.unstack().reindex(ORDER)
    im = a_.imshow(hm.values, aspect="auto", cmap="magma", vmin=GPUS[0], vmax=GMAX)
    a_.set_xticks(range(hm.shape[1])[1::4], [f"{v:g}" for v in hm.columns][1::4])
    a_.set(xlabel="speed $v$ [m/s]", title=f"mode {mode}: oracle GPU")
ax[0].set_yticks(YT, [str(NV[i]) for i in YT]); ax[0].set_ylabel("inputs (by $N_{vis}$)")
fig.colorbar(im, ax=ax, fraction=.05, pad=.02).set_label("GPU [MHz]")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=plot_style.FIG2, sharey=True)
for a_, mode in zip(ax, "AB"):
    R = globals()[f"R_{mode}"]
    m = R.groupby(level=1)[["max", "governor", "input-only", "speed-only", "oracle"]].mean()
    for c in m:
        a_.plot(m.index, m[c], "o-", ms=3, label=c)
    a_.set(xlabel="speed $v$ [m/s]", title=f"mode {mode}")
    a_.grid(alpha=.3)
ax[0].set_ylabel("average power [W]")
h, l = ax[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, frameon=False, columnspacing=1, handletextpad=.3)
plt.tight_layout(rect=(0, 0, 1, .8)); plt.show()

**结论**：
- 最优 GPU 频率在（输入，速度）平面上变化很大，没有一个固定设置能同时覆盖所有情况。
- 只看速度、或只看输入的策略，都必须按最坏情况选频率，节省的功率都比 oracle 少。
- **同时利用输入和速度信息**才能拿到全部收益。
- **两种调用方式的收益差别很大**（n_act=8，dq=2 m）：
  - 连续查询（A）：oracle 比最高频省约 38%，比默认 governor 省约 30%（9.2 W 对 13.1 W）。
  - 事件触发（B）：oracle 比最高频只省约 18%，比默认 governor 只省约 7%（9.0 W 对 9.6 W）。原因是大部分时间在空闲，空闲功率占了平均功率的大头。
  - 所以**连续查询是这个 objective 最有说服力的场景**。事件触发更适合作为"调用方式"这个维度的对比。
- 注意：默认 governor 的平均功率只算了它能满足约束的点，在高速时偏乐观。

## Obs 7：如果速度也能自由选，问题会退化

如果让速度也变成决策变量，目标改成**任务总计算能耗**（走完 S 米的计算能耗 = P̄·S/v），会怎样？下面算一下每个输入下最优的速度。

In [ ]:
rows = []
for mode in "AB":
    R = globals()[f"R_{mode}"]
    jpm = (R.oracle / R.index.get_level_values(1)).rename("J_per_m")       # compute energy per metre
    for k, d in jpm.groupby(level=0):
        vbest = d.idxmin()[1]
        vmax = d.index.get_level_values(1).max()
        rows.append(dict(mode=mode, inp=k, v_best=vbest, v_max_feasible=vmax))
r = pd.DataFrame(rows)
for mode, d in r.groupby("mode"):
    print(f"mode {mode}: {(d.v_best == d.v_max_feasible).mean() * 100:.0f}% of inputs have "
          f"energy-optimal speed = fastest feasible speed")

**结论**：
- 把速度也放开时，计算能耗最低的速度绝大多数情况下（连续查询 86%，事件触发 97% 的输入）就是最高可行速度，对应的频率也接近最高频。也就是说，"跑得越快越省"，问题退化了。
- 再加上机械能耗、用之前的 P_mech 模型也一样会退化，这一点之前已经验证过。
- 所以把速度当作**任务给定**的量更合理，也更符合实际：速度由规划器根据路况、人流等决定。

## 可以作为 Objective 的几种定义

### O1（推荐）：同时利用输入和速度的 DVFS，计算能耗最小

**已知**：任务输入 x = (分辨率, 图片数, N_txt, n_act)，速度 v，D，查询方式（连续，或每 dq 米查询一次）。

min_{f ∈ F_CPU × F_GPU}  P̄(f; x, v)

约束：
- 安全：L_q(f; x, n_act) ≤ τ(v) = (D − margin − v²/2a)/v − t_act
- 事件触发时还要满足查询频率：L_q(f; x, n_act) ≤ T_q = dq / v

其中 P̄ 是平均功率：连续查询时 P̄ = E_req / L；事件触发时 P̄ = [E_req + P_idle(T_q − L)] / T_q。

**数据支撑**：
- Obs 1：输入让延迟相差一个数量级以上。
- Obs 2：GPU 频率是有效的旋钮，CPU 应该放低。
- Obs 3：连续查询和事件触发的最优点相反，必须指定调用方式。
- Obs 4：速度通过 τ(v) 给出延迟上限。
- Obs 5：默认 governor 两头都不好。
- Obs 6：oracle 比最高频和默认 governor 省得多，只看一个维度的策略省不全。
- Obs 7：v 必须是给定的，否则问题退化。

### O2（O1 的对偶）：给定功率预算，最大化安全速度

max_f v*(f; x)  s.t.  P̄(f; x) ≤ P_budget

适用场景：电池或散热限制了功率，规划器需要知道"当前输入下最快能开多快"。数据就是 Obs 4 的速度上限和 Obs 3 的功率曲线组成的 Pareto 前沿。

### O3（不推荐）：速度和频率一起优化，最小化任务总能耗

min_{v, f} P̄(f)·S/v（还可以加上机械能耗）。按 Obs 7，最优解会落在最高可行速度加接近最高频，没有研究价值，只适合作为说明"为什么要把速度当作给定量"的反例。

### 其他可选的方向
- **延迟分布**：现在用 p95，可以换成要求违反安全约束的概率 ≤ ε 的形式。
- **调用方式本身可以作为决策变量**：用 Obs 3 的数据比较连续查询和事件触发，以及查询间隔 dq 选多大。